# Q-factorisation on Gridworld Maze

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import time
from copy import deepcopy

from sklearn.decomposition import PCA

# Resolve repository src path robustly when running notebook in-place.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / 'src').exists():
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / 'src'))

from environments.maze_discrete import MazeGridWorld, MazeGoalWrapper
from utils import TrajectoryReplayBufferDiscrete, evaluate_policy, set_seed, build_goal_batch
from visualisations import plot_policy_rollouts, plot_q_diagnostics, plot_full_embedding_dashboard_html
from loss_functions import repulsion_loss_to_memory, sigreg_loss, orthogonal_loss, ewc_regulariser_loss, weight_regulariser_loss
from networks import snapshot_named_parameters


DEVICE = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print('Using device:', DEVICE)


In [ ]:
import torch.nn.functional as F
def get_base_env(env):
    return env.unwrapped

def collect_valid_states_fourrooms(env):

    base_env = get_base_env(env)

    if not hasattr(base_env, "_free_cells"):
        raise RuntimeError("Base env does not expose _free_cells.")

    coords = np.asarray(base_env._free_cells, dtype=np.int32)      # [N, 2]
    states = coords.astype(np.float32)                             # obs == (x, y)

    return states, coords

def shared_pca_projection(emb_before, emb_after, n_components=2):
    X = np.concatenate([emb_before, emb_after], axis=0)
    pca = PCA(n_components=n_components)
    Xp = pca.fit_transform(X)
    Z_before = Xp[:emb_before.shape[0]]
    Z_after = Xp[emb_before.shape[0]:]
    return Z_before, Z_after, pca

def plot_before_after(Z_before, Z_after, label_before, label_after, title):
    plt.figure(figsize=(7, 6))
    plt.scatter(Z_before[:, 0], Z_before[:, 1], s=25, alpha=0.7, label=label_before)
    plt.scatter(Z_after[:, 0], Z_after[:, 1], s=25, alpha=0.7, label=label_after)
    plt.xlabel("PC 1")
    plt.ylabel("PC 2")
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()

def estimate_fisher_diag(
    model,
    target_model,
    replay_buffer,
    goal,
    num_actions,
    device=DEVICE,
    gamma=0.99,
    batch_size=256,
    n_batches=64,
    prefix_filter="sa_encoder",
    use_success_only=False,
):
    """
    Estimate diagonal Fisher for EWC via TD loss on replay samples.

    If use_success_only is True, this assumes replay_buffer has a
    .sample_success(batch_size) method; otherwise it falls back to .sample.
    """

    model.eval()
    target_model.eval()

    fisher_diag = {}

    # Initialise Fisher entries for selected parameters
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue

        if prefix_filter is None:
            fisher_diag[name] = torch.zeros_like(p, device=device)
        elif isinstance(prefix_filter, str):
            if name.startswith(prefix_filter):
                fisher_diag[name] = torch.zeros_like(p, device=device)
        else:
            if any(name.startswith(pref) for pref in prefix_filter):
                fisher_diag[name] = torch.zeros_like(p, device=device)

    if len(fisher_diag) == 0:
        raise ValueError("No parameters matched prefix_filter for Fisher estimation.")

    # Accumulate squared gradients
    num_batches_used = 0

    for _ in range(n_batches):
        if use_success_only and hasattr(replay_buffer, "sample_success"):
            batch = replay_buffer.sample_success(batch_size)
        else:
            batch = replay_buffer.sample(batch_size)

        obs_t = batch.obs          # [B, obs_dim]
        act_t = batch.actions.long()   # [B, 1]
        rew_t = batch.rewards      # [B, 1]
        next_obs_t = batch.next_obs
        term_t = batch.terminated
        trunc_t = batch.truncated
        done_t = torch.clamp(term_t + trunc_t, 0.0, 1.0)

        B = obs_t.shape[0]
        goal_batch = build_goal_batch(goal, B, device)

        with torch.no_grad():
            next_q_vals = target_model.q_val_for_argmax_action(next_obs_t, goal_batch)
            next_q = next_q_vals.max(dim=-1, keepdim=True).values
            target = rew_t + gamma * (1.0 - done_t) * next_q

        act_onehot = torch.nn.functional.one_hot(
            act_t.squeeze(-1),
            num_classes=num_actions
        ).float()

        current_q = model(obs_t, act_onehot, goal_batch)
        td_loss = torch.nn.functional.mse_loss(current_q, target)

        model.zero_grad(set_to_none=True)
        td_loss.backward()

        for name, p in model.named_parameters():
            if name in fisher_diag and p.grad is not None:
                fisher_diag[name] += p.grad.detach().pow(2)

        num_batches_used += 1

    if num_batches_used == 0:
        raise RuntimeError("No batches used in Fisher estimation.")

    # Average over batches
    for name in fisher_diag:
        fisher_diag[name] /= float(num_batches_used)

    # Per-parameter normalisation to stabilise scale
    eps = 1e-8
    for name, F in fisher_diag.items():
        mean_val = F.mean()
        if mean_val > 0:
            fisher_diag[name] = F / (mean_val + eps)

    scale = 10.0  # try 10, 50, etc.
    for name in fisher_diag:
        fisher_diag[name] = fisher_diag[name] * scale

    model.train()
    target_model.train()

    return fisher_diag


def extract_mean_sa_embedding(
    q_network,
    buffer,
    num_actions,
    batch_size=256,
    device=None,
    as_numpy=True,
):
    """
    Mean SA embedding over a sampled batch from replay buffer.
    Useful as a compact one-vector-per-task summary for visualisation.
    Returns shape [rep_dim].
    """
    if device is None:
        device = next(q_network.parameters()).device

    n_available = len(buffer)
    if n_available == 0:
        raise ValueError("Replay buffer is empty; cannot extract SA embeddings.")

    probe_bs = min(batch_size, n_available)

    q_network.eval()
    with torch.no_grad():
        probe_batch = buffer.sample(probe_bs)
        obs_t = probe_batch.obs.to(device)                              # [B, obs_dim]
        act_idx = probe_batch.actions.long().squeeze(-1).to(device)    # [B]
        act_onehot = F.one_hot(act_idx, num_classes=num_actions).float()

        phi_sa = q_network.encode_state_action(obs_t, act_onehot)      # [B, D]
        mean_sa_embedding = phi_sa.mean(dim=0)                         # [D]

    if as_numpy:
        return mean_sa_embedding.detach().cpu().numpy()
    return mean_sa_embedding.detach().cpu()


def extract_fixed_probe_sa_embedding(
    q_network,
    obs_probe,
    act_probe_idx,
    num_actions,
    device=None,
    as_numpy=True,
):
    """
    SA embedding for one fixed (obs, act) probe.
    Useful for comparing how the same input is represented across tasks.
    Returns shape [rep_dim].
    """
    if device is None:
        device = next(q_network.parameters()).device

    if not torch.is_tensor(obs_probe):
        obs_probe = torch.tensor(obs_probe, dtype=torch.float32, device=device)
    else:
        obs_probe = obs_probe.to(device).float()

    obs_probe = obs_probe.unsqueeze(0)  # [1, obs_dim]

    act_probe = F.one_hot(
        torch.tensor([act_probe_idx], device=device),
        num_classes=num_actions,
    ).float()  # [1, action_dim]

    q_network.eval()
    with torch.no_grad():
        phi_sa = q_network.encode_state_action(obs_probe, act_probe)   # [1, D]
        phi_sa = phi_sa.squeeze(0)

    if as_numpy:
        return phi_sa.detach().cpu().numpy()
    return phi_sa.detach().cpu()


def extract_sa_batch_for_isotropy(
    q_network,
    buffer,
    num_actions,
    batch_size=1024,
    device=None,
    as_numpy=True,
):
    """
    Full batch of SA embeddings for isotropy diagnostics.
    This is the one that matters for SIGReg checks.
    Returns shape [B, rep_dim].
    """
    if device is None:
        device = next(q_network.parameters()).device

    n_available = len(buffer)
    if n_available == 0:
        raise ValueError("Replay buffer is empty; cannot extract SA batch.")

    probe_bs = min(batch_size, n_available)

    q_network.eval()
    with torch.no_grad():
        probe_batch = buffer.sample(probe_bs)
        obs_t = probe_batch.obs.to(device)
        act_idx = probe_batch.actions.long().squeeze(-1).to(device)
        act_onehot = F.one_hot(act_idx, num_classes=num_actions).float()

        phi_sa = q_network.encode_state_action(obs_t, act_onehot)      # [B, D]

    if as_numpy:
        return phi_sa.detach().cpu().numpy()
    return phi_sa.detach().cpu()

In [ ]:
class FactorisedDQN_QNetwork(nn.Module):
    def __init__(
        self,
        obs_dim: int,
        num_actions: int,
        goal_dim: int = 2,
        hidden_dim: int = 128,
        rep_dim: int = 64,
    ):
        super().__init__()
        self.obs_dim = obs_dim
        self.num_actions = num_actions
        self.action_dim = num_actions
        self.goal_dim = goal_dim
        self.rep_dim = rep_dim

        self.sa_encoder = nn.Sequential(
            nn.Linear(obs_dim + self.action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
        )

        self.goal_encoder = nn.Sequential(
            nn.Linear(goal_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, rep_dim),
        )

    def encode_goal(self, goal: torch.Tensor) -> torch.Tensor:
        psi = self.goal_encoder(goal)
        psi = torch.tanh(psi)
        psi = F.normalize(psi, p=2, dim=-1, eps=1e-8)
        return psi

    def encode_state_action(self, obs: torch.Tensor, act: torch.Tensor) -> torch.Tensor:
        sa = torch.cat([obs, act], dim=-1)
        phi = self.sa_encoder(sa)
        phi = torch.tanh(phi)
        phi = F.normalize(phi, p=2, dim=-1, eps=1e-8)
        return phi

    def forward(
        self,
        obs: torch.Tensor,
        act: torch.Tensor,
        goal: torch.Tensor,
    ) -> torch.Tensor:
        phi_sa = self.encode_state_action(obs, act)      # [B, D]
        psi_z = self.encode_goal(goal)                   # [B, D]
        q_vals = (phi_sa * psi_z).sum(dim=-1, keepdim=True)  # [B, 1]
        return q_vals

    def q_val_for_argmax_action(self, obs: torch.Tensor, goal: torch.Tensor) -> torch.Tensor:
        B = obs.shape[0]
        A = self.num_actions

        act_onehot = F.one_hot(
            torch.arange(A, device=obs.device),
            num_classes=self.action_dim
        ).float()                                        # [A, A]
        act_onehot = act_onehot.unsqueeze(0).expand(B, -1, -1)   # [B, A, A]

        obs_rep = obs.unsqueeze(1).expand(-1, A, -1)             # [B, A, obs_dim]
        obs_flat = obs_rep.reshape(B * A, self.obs_dim)          # [B*A, obs_dim]
        act_flat = act_onehot.reshape(B * A, self.action_dim)    # [B*A, A]

        phi_sa = self.encode_state_action(obs_flat, act_flat)    # [B*A, D]
        phi_sa = phi_sa.view(B, A, self.rep_dim)                 # [B, A, D]

        psi_z = self.encode_goal(goal)                           # [B, D]
        psi_rep = psi_z.unsqueeze(1).expand(B, A, -1)           # [B, A, D]

        q_vals = (phi_sa * psi_rep).sum(dim=-1)                 # [B, A]
        return q_vals
    
    def forward_with_task_embedding(
        self,
        obs: torch.Tensor,
        act: torch.Tensor,
        task_embedding: torch.Tensor,
        normalize_embedding: bool = False,
    ) -> torch.Tensor:
        """
        Compute Q(obs, act | psi) directly from a provided task embedding.

        obs: [B, obs_dim]
        act: [B, action_dim]
        task_embedding: [B, rep_dim] or [rep_dim]
        """
        phi_sa = self.encode_state_action(obs, act)  # [B, D]

        if task_embedding.dim() == 1:
            task_embedding = task_embedding.unsqueeze(0).expand(obs.shape[0], -1)

        psi_z = (
            F.normalize(task_embedding, p=2, dim=-1, eps=1e-8)
            if normalize_embedding
            else task_embedding
        )

        q_vals = (phi_sa * psi_z).sum(dim=-1, keepdim=True)   # [B, 1]
        return q_vals
    
    def q_val_for_argmax_action_from_embedding(
        self,
        obs: torch.Tensor,
        task_embedding: torch.Tensor,
        normalize_embedding: bool = False,
    ) -> torch.Tensor:
        """
        Compute Q-values for all actions using a provided task embedding.

        obs: [B, obs_dim]
        task_embedding: [B, rep_dim] or [rep_dim]
        returns: [B, A]
        """
        B = obs.shape[0]
        A = self.num_actions

        act_onehot = F.one_hot(
            torch.arange(A, device=obs.device),
            num_classes=self.action_dim
        ).float()                                             # [A, A]
        act_onehot = act_onehot.unsqueeze(0).expand(B, -1, -1)  # [B, A, A]

        obs_rep = obs.unsqueeze(1).expand(-1, A, -1)          # [B, A, obs_dim]
        obs_flat = obs_rep.reshape(B * A, self.obs_dim)       # [B*A, obs_dim]
        act_flat = act_onehot.reshape(B * A, self.action_dim) # [B*A, A]

        phi_sa = self.encode_state_action(obs_flat, act_flat) # [B*A, D]
        phi_sa = phi_sa.view(B, A, self.rep_dim)              # [B, A, D]

        if task_embedding.dim() == 1:
            task_embedding = task_embedding.unsqueeze(0).expand(B, -1)

        psi_z = (
            F.normalize(task_embedding, p=2, dim=-1, eps=1e-8)
            if normalize_embedding
            else task_embedding
        )

        psi_rep = psi_z.unsqueeze(1).expand(B, A, -1)         # [B, A, D]
        q_vals = (phi_sa * psi_rep).sum(dim=-1)               # [B, A]
        return q_vals

In [ ]:
MAZE_LAYOUT_30 = [
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
    [1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1],
    [1,0,1,1,1,0,1,0,1,1,1,0,1,0,1,1,1,0,1,0,1,1,1,0,1,0,1,1,0,1],
    [1,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1],
    [1,0,1,0,1,1,1,1,1,0,1,1,1,1,1,0,1,1,1,1,1,0,1,1,1,1,1,0,1,1],
    [1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,1],
    [1,0,1,1,1,1,1,0,1,1,1,0,1,0,1,1,1,0,1,0,1,1,1,1,1,0,1,1,0,1],
    [1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1],
    [1,1,1,1,1,0,1,1,1,0,1,1,1,0,1,0,1,1,1,1,1,1,1,0,1,1,1,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1],
    [1,0,1,0,1,1,1,0,1,1,1,0,1,1,1,0,1,0,1,1,1,0,1,1,1,1,0,1,0,1],
    [1,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,0,1,0,0,0,1],
    [1,0,1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1,0,1,1,1,1,0,1,1,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,1,0,1],
    [1,1,1,0,1,1,1,0,1,1,1,0,1,1,1,1,1,0,1,1,1,1,0,1,1,1,0,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,1],
    [1,0,1,1,1,0,1,1,1,0,1,1,1,0,1,0,1,1,1,1,1,1,0,1,0,1,1,1,0,1],
    [1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,1],
    [1,1,1,1,1,0,1,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,1,1,1,1,1,1,0,1],
    [1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,1],
    [1,0,1,0,1,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,1,0,1,0,1,1,1,1,0,1],
    [1,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,1,0,1],
    [1,0,1,1,1,1,1,0,1,1,1,0,1,0,1,1,1,1,1,0,1,1,1,1,1,0,1,1,0,1],
    [1,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,1],
    [1,1,1,1,1,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,1,1,1,0,1,1,1,1,0,1],
    [1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,1,0,1],
    [1,0,1,0,1,1,1,0,1,1,1,1,1,0,1,0,1,1,1,1,1,0,1,1,1,1,0,1,0,1],
    [1,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1],
    [1,0,1,1,1,0,1,1,1,1,1,0,1,1,1,0,1,0,1,1,1,1,1,1,0,1,1,1,0,1],
    [1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1],
]

def make_env(goal=(28, 28), slip_prob=0.00, max_horizon=1000):
    base = MazeGridWorld(
        maze=MAZE_LAYOUT_30,
        max_episode_steps=max_horizon,
    )
    env = MazeGoalWrapper(
        base,
        goal_position=goal,
        goal_reward=1.0,
        step_reward=0.0,
        slip_prob=slip_prob,
        reward_mode="simple",
    )
    return env

env = make_env(goal=(28, 28))
obs, info = env.reset()

img = env.unwrapped.render()
goal = env.goal_position

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.scatter(
    goal[0] * 40 + 20,
    goal[1] * 40 + 20,
    c="lime",
    s=180,
    marker="*",
    edgecolors="black",
)
plt.title(f"Maze with goal at {goal}")
plt.axis("off")
plt.show()

In [ ]:
BUFFER_CAPACITY = 100000
GOAL = (9, 9)
LR = float(1e-3)

env = make_env(goal=GOAL)
obs_dim = env.observation_space.shape[0]
num_actions = env.action_space.n
task_embedding_memory = []


# Factorised Q-network instead of plain DQN_QNetwork
# Goal is 2-D (grid coordinates), so goal_dim=2
q_net = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)

q_target = FactorisedDQN_QNetwork(
    obs_dim=obs_dim,
    num_actions=num_actions,
    goal_dim=2,
    hidden_dim=128,
    rep_dim=64,
).to(DEVICE)

q_target.load_state_dict(q_net.state_dict())
for p in q_target.parameters():
    p.requires_grad_(False)


def dqn_train(
    seed: int = 42,
    q_network=q_net,
    q_target_network=q_target,
    env=env,
    buffer_capacity=BUFFER_CAPACITY,
    lr=LR,
    obs_dim=obs_dim,
    device=DEVICE,
    total_steps=150000,
    warmup_steps=5000,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    eps_start=1.0,
    eps_end=0.05,
    eps_decay_steps=100000,
    train_freq=4,
    goal=GOAL,
    params=None,
    regulariser=None,
    reg_alpha=500,
    embedding_memory=task_embedding_memory,
    reference_params=None,
    fisher_diag=None,
    sa_reg_prefix_filter="sa_encoder",
    sigreg=1,
):
    set_seed(seed)

    if params is None:
        opt = optim.Adam([
            {"params": q_network.sa_encoder.parameters(), "lr": lr},
            {"params": q_network.goal_encoder.parameters(), "lr": lr},
        ])
    else:
        opt = optim.Adam(params, lr=lr)

    buffer = TrajectoryReplayBufferDiscrete(buffer_capacity, obs_dim, 1, device=device)

    goal_arr = np.array(goal, dtype=np.float32)
    goal_t_single = torch.tensor(goal_arr, dtype=torch.float32, device=device).unsqueeze(0)

    obs, _ = env.reset()
    global_step = 0
    eval_returns = []
    eval_returns_time = []
    start_time = time.perf_counter()
    min_steps = None
    min_time = None
    ortho_loss = torch.tensor(0.0, device=device)
    sigreg_loss_val = torch.tensor(0.0, device=device)
    weight_loss = torch.tensor(0.0, device=device)
    ewc_loss = torch.tensor(0.0, device=device)
    loss = torch.tensor(0.0, device=device)

    ep_obs, ep_actions, ep_rewards, ep_next_obs, ep_terminated, ep_truncated = (
        [], [], [], [], [], []
    )

    num_actions = env.action_space.n

    while global_step < total_steps:
        frac = min(1.0, global_step / eps_decay_steps)
        eps = eps_start + frac * (eps_end - eps_start)

        if np.random.random() < eps:
            action = env.action_space.sample()
        else:
            obs_t = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)
            with torch.no_grad():
                q_vals = q_network.q_val_for_argmax_action(obs_t, goal_t_single)  # [1, A]
                action = int(q_vals.argmax(dim=-1).item())

        next_obs, rew, term, trunc, _ = env.step(action)
        done = term or trunc

        ep_obs.append(obs.copy())
        ep_actions.append(action)
        ep_rewards.append(float(rew))
        ep_next_obs.append(next_obs.copy())
        ep_terminated.append(float(term))
        ep_truncated.append(float(trunc))

        obs = next_obs
        global_step += 1

        if done:
            episode = {
                "obs": ep_obs,
                "actions": ep_actions,
                "rewards": ep_rewards,
                "next_obs": ep_next_obs,
                "terminated": ep_terminated,
                "truncated": ep_truncated,
            }
            buffer.add_episode(episode)
            ep_obs, ep_actions, ep_rewards, ep_next_obs, ep_terminated, ep_truncated = (
                [], [], [], [], [], []
            )
            obs, _ = env.reset()

        if len(buffer) >= warmup_steps and global_step % train_freq == 0:
            batch = buffer.sample(batch_size)

            obs_t = batch.obs                    # [B, obs_dim]
            act_t = batch.actions.long()         # [B, 1]
            rew_t = batch.rewards                # [B, 1]
            next_obs_t = batch.next_obs          # [B, obs_dim]
            term_t = batch.terminated            # [B, 1]
            trunc_t = batch.truncated            # [B, 1]
            done_t = torch.clamp(term_t + trunc_t, 0.0, 1.0)

            goal_batch = goal_t_single.expand(obs_t.shape[0], -1)   # [B, goal_dim]
            B = obs_t.shape[0]

            with torch.no_grad():
                next_q_vals = q_target_network.q_val_for_argmax_action(next_obs_t, goal_batch)  # [B, A]
                next_q = next_q_vals.max(dim=-1, keepdim=True).values                            # [B, 1]
                target = rew_t + gamma * (1.0 - done_t) * next_q                                 # [B, 1]

            act_onehot = F.one_hot(
                act_t.squeeze(-1),
                num_classes=num_actions
            ).float()                                                                            # [B, A]

            current_q = q_network(obs_t, act_onehot, goal_batch)                                 # [B, 1]
            td_loss = F.mse_loss(current_q, target)

            if sigreg is not None:
                act_onehot_all = F.one_hot(
                    torch.arange(num_actions, device=device),
                    num_classes=num_actions
                ).float()                                                                            # [A, A]
                act_onehot_all = act_onehot_all.unsqueeze(0).expand(B, -1, -1)                      # [B, A, A]

                obs_rep = obs_t.unsqueeze(1).expand(-1, num_actions, -1)                            # [B, A, obs_dim]
                obs_flat = obs_rep.reshape(B * num_actions, obs_dim)                                 # [B*A, obs_dim]
                act_flat = act_onehot_all.reshape(B * num_actions, num_actions)                      # [B*A, A]

                phi_all = q_network.encode_state_action(obs_flat, act_flat)                          # [B*A, D]
                sigreg_loss_val = sigreg_loss(phi_all)

            if reference_params is not None:
                weight_loss = weight_regulariser_loss(q_network, reference_params=reference_params, prefix_filter=sa_reg_prefix_filter)

            if fisher_diag is not None:
                ewc_loss = ewc_regulariser_loss(q_network, reference_params=reference_params, fisher_diag=fisher_diag, prefix_filter=sa_reg_prefix_filter)

            if regulariser is not None and regulariser == 'repulsion':
                #ortho_loss = orthogonal_loss(q_network, goal_t_single, embedding_memory, device)
                loss = td_loss + reg_alpha * ortho_loss + 1 * sigreg_loss_val + 0.1 * weight_loss + 10 * ewc_loss
            else:
                loss = td_loss + 0.1 * sigreg_loss_val + 10 * ewc_loss

            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_network.parameters(), 10.0)
            opt.step()

            for p, p_tgt in zip(q_network.parameters(), q_target_network.parameters()):
                p_tgt.data.mul_(1.0 - tau).add_(tau * p.data)
        
        if global_step % 1000 == 0 and reference_params is not None:
            delta_means = []
            for name, p in q_network.named_parameters():
                if name in reference_params:
                    delta_sq = (p.detach() - reference_params[name]).pow(2)
                    delta_means.append(delta_sq.mean().item())

            if len(delta_means) > 0:
                print("Mean squared deltas (first few):", delta_means[:5])

        if global_step % 1000 == 0:
            eval_env = make_env(goal)
            goal_eval_arr = np.array(goal, dtype=np.float32)
            goal_eval_t = torch.tensor(goal_eval_arr, dtype=torch.float32, device=device).unsqueeze(0)

            def eval_policy(o):
                o_t = torch.tensor(o, dtype=torch.float32, device=device).unsqueeze(0)
                with torch.no_grad():
                    q_vals = q_network.q_val_for_argmax_action(o_t, goal_eval_t)  # [1, A]
                    return int(q_vals.argmax(dim=-1).item())

            mean_ret, mean_len = evaluate_policy(eval_env, eval_policy, episodes=8)
            eval_time = time.perf_counter() - start_time
            eval_returns_time.append((eval_time, mean_ret))
            eval_returns.append((global_step, mean_ret))
            print(
                f"[DQN-factorised] step={global_step:7d} | eps={eps:.3f} "
                f"| eval_return={mean_ret:.3f} | eval_len={mean_len:.1f}"
                f"| Ortho loss={ortho_loss.item():.3f} | SigReg loss={sigreg_loss_val.item():.3f} | Loss={loss.item():.3f}"
                f"| Weight loss={weight_loss.item():.10f} | EWC loss={ewc_loss.item():.10f}"
            )
            if mean_ret >= 0.99 and min_steps is None:
                min_steps = global_step
                min_time = eval_time
                print(f"Good policy achieved at step {global_step} with mean return {mean_ret:.3f}")
            eval_env.close()

    goal_tensor = torch.tensor(np.array(goal, dtype=np.float32), dtype=torch.float32, device=device).unsqueeze(0)
    with torch.no_grad():
        psi_z = q_network.encode_goal(goal_tensor)   # [1, rep_dim]
        task_embedding = psi_z.squeeze(0).cpu().numpy()

    obs_probe_np = np.array([5.0, 5.0], dtype=np.float32)

    base_env = env.unwrapped if hasattr(env, "unwrapped") else env
    env_action_names = getattr(base_env, "action_names", ["Up", "Down", "Left", "Right"])
    act_probe_idx = env_action_names.index("Up")

    sa_embedding_mean = extract_mean_sa_embedding(
        q_network=q_network,
        buffer=buffer,
        num_actions=num_actions,
        batch_size=256,
        device=device,
        as_numpy=True,
    )

    sa_embedding_fixed = extract_fixed_probe_sa_embedding(
        q_network=q_network,
        obs_probe=obs_probe_np,
        act_probe_idx=act_probe_idx,
        num_actions=num_actions,
        device=device,
        as_numpy=True,
    )

    sa_batch_final = extract_sa_batch_for_isotropy(
        q_network=q_network,
        buffer=buffer,
        num_actions=num_actions,
        batch_size=1024,
        device=device,
        as_numpy=True,
    )

    env.close()
    return (
        q_network,
        q_target_network,
        eval_returns,
        eval_returns_time,
        min_steps if 'min_steps' in locals() else None,
        min_time if 'min_time' in locals() else None,
        task_embedding,
        sa_embedding_mean,
        sa_embedding_fixed,
        sa_batch_final,
        buffer
    )

def plot_eval_results(eval_first, eval_time_first, min_steps_first, min_time_first):
    if eval_first and eval_time_first:
        xs_steps, ys_steps = zip(*eval_first)
        xs_time, ys_time = zip(*eval_time_first)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

        ax1.plot(xs_steps, ys_steps)
        if min_steps_first is not None:
            ax1.axvline(min_steps_first, color='red', linestyle='--', label='Good policy achieved')
            ax1.annotate(
                f"Good policy achieved at step {min_steps_first}",
                xy=(min_steps_first, 0.99),
                xytext=(min_steps_first + 5000, 0.5),
                arrowprops=dict(arrowstyle="->", color='red'),
                color='red',
            )
        ax1.set_xlabel("Environment steps")
        ax1.set_ylabel("Mean episodic return")
        ax1.set_title("Q = φ(s,a)ᵀψ(z) on FourRooms (discrete) – steps")
        ax1.grid(alpha=0.25)
        
        ax2.plot(xs_time, ys_time)
        if min_time_first is not None:
            ax2.axvline(min_time_first, color='red', linestyle='--', label='Good policy achieved')
            ax2.annotate(
                f"Good policy achieved at time {min_time_first:.2f}s",
                xy=(min_time_first, 0.99),
                xytext=(min_time_first + 5.0, 0.5),
                arrowprops=dict(arrowstyle="->", color='red'),
                color='red',
            )
        ax2.set_xlabel("Evaluation time (s)")
        ax2.set_ylabel("Mean episodic return")
        ax2.set_title("Q = φ(s,a)ᵀψ(z) on FourRooms (discrete) – time")
        ax2.grid(alpha=0.25)

        plt.tight_layout()
        plt.show()

def print_goal_embedding_similarity(task_embedding_memory, goal_labels=None, decimals=3):
    if len(task_embedding_memory) == 0:
        print("Goal embedding memory is empty.")
        return

    M = np.stack(task_embedding_memory, axis=0)  # [N, D]
    M = M / (np.linalg.norm(M, axis=1, keepdims=True) + 1e-8)
    sim = M @ M.T  # cosine similarity matrix, [N, N]

    if goal_labels is None:
        goal_labels = [f"g{i}" for i in range(len(task_embedding_memory))]

    print("\nGoal embedding cosine similarity matrix:")
    header = " " + " ".join([f"{str(label):>10s}" for label in goal_labels])
    print(header)

    for i, row in enumerate(sim):
        row_str = " ".join([f"{x:10.{decimals}f}" for x in row])
        print(f"{str(goal_labels[i]):>8s} {row_str}")

    if len(task_embedding_memory) > 1:
        upper = sim[np.triu_indices(len(task_embedding_memory), k=1)]
        print(f"\nOff-diagonal mean similarity: {upper.mean():.{decimals}f}")
        print(f"Off-diagonal min similarity: {upper.min():.{decimals}f}")
        print(f"Off-diagonal max similarity: {upper.max():.{decimals}f}")



In [ ]:
def visualise_q_table(goal, q_network, eval_returns=None, task_embedding=None):
    q_network.eval()
    eval_env_first = make_env(goal=goal)

    base_env = eval_env_first.unwrapped if hasattr(eval_env_first, "unwrapped") else eval_env_first
    env_action_names = getattr(base_env, "action_names", ["Up", "Down", "Left", "Right"])

    if task_embedding is not None:
        if not torch.is_tensor(task_embedding):
            task_embedding = torch.tensor(task_embedding, dtype=torch.float32, device=DEVICE)
        else:
            task_embedding = task_embedding.to(DEVICE).float()

    def dqn_policy_fn(obs, q_network, goal, task_embedding=None):
        obs_t = torch.tensor(obs, dtype=torch.float32, device=DEVICE).unsqueeze(0)

        with torch.no_grad():
            if task_embedding is None:
                goal_arr = np.array(goal, dtype=np.float32)
                goal_t = torch.tensor(goal_arr, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                q_vals = q_network.q_val_for_argmax_action(obs_t, goal_t)
            else:
                q_vals = q_network.q_val_for_argmax_action_from_embedding(
                    obs_t,
                    task_embedding,
                    normalize_embedding=False,
                )

            action = int(q_vals.argmax(dim=-1).item())

        return action

    def dqn_value_fn(obs_batch, q_network, goal, task_embedding=None):
        obs_t = torch.tensor(obs_batch, dtype=torch.float32, device=DEVICE)
        B = obs_t.shape[0]

        with torch.no_grad():
            if task_embedding is None:
                goal_arr = np.array(goal, dtype=np.float32)
                goal_t = torch.tensor(goal_arr, dtype=torch.float32, device=DEVICE).unsqueeze(0)
                goal_batch = goal_t.expand(B, -1)
                q_vals = q_network.q_val_for_argmax_action(obs_t, goal_batch)
            else:
                q_vals = q_network.q_val_for_argmax_action_from_embedding(
                    obs_t,
                    task_embedding,
                    normalize_embedding=False,
                )

            q_vals = q_vals.cpu().numpy()

        return q_vals

    plot_policy_rollouts(
        env=eval_env_first,
        policy_fn=lambda obs: dqn_policy_fn(
            obs,
            q_network=q_network,
            goal=goal,
            task_embedding=task_embedding,
        ),
        goal_pos=goal,
        eval_episodes=8,
        n_cols=4,
        is_discrete=True,
        step_point_size=12,
        start_size=60,
        end_size=50,
        goal_size=130,
        arrow_width=0.01,
    )

    plot_q_diagnostics(
        env=eval_env_first,
        value_fn=lambda obs_batch: dqn_value_fn(
            obs_batch,
            q_network=q_network,
            goal=goal,
            task_embedding=task_embedding,
        ),
        actor_fn=None,
        is_discrete=True,
        num_actions=eval_env_first.action_space.n,
        action_names=env_action_names,
        goal_pos=goal,
        eval_returns=eval_returns,
    )

    eval_env_first.close()
    q_network.train()

    return q_network

In [ ]:

def visualise_embeddings(goal, q_network):
    eval_env_first = make_env(goal=goal)
    base_goal = np.array(goal, dtype=np.float32)
    base_goal_t = torch.tensor(base_goal, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    states_base, coords_base = collect_valid_states_fourrooms(eval_env_first)
    obs_base_t = torch.tensor(states_base, dtype=torch.float32, device=DEVICE)

    q_network.eval()
    with torch.no_grad():
        # No encode_state in the new class, so remove phi_s_base analysis
        psi_z_base = q_network.encode_goal(base_goal_t).cpu().numpy()  # [1, D]

        N = obs_base_t.shape[0]
        A = q_network.num_actions

        act_onehot = F.one_hot(
            torch.arange(A, device=DEVICE),
            num_classes=q_network.action_dim
        ).float()                                                      # [A, A]

        obs_rep = obs_base_t.unsqueeze(1).expand(-1, A, -1)            # [N, A, obs_dim]
        act_rep = act_onehot.unsqueeze(0).expand(N, -1, -1)            # [N, A, A]

        obs_flat = obs_rep.reshape(N * A, q_network.obs_dim)           # [N*A, obs_dim]
        act_flat = act_rep.reshape(N * A, q_network.action_dim)        # [N*A, A]

        phi_sa_base = q_network.encode_state_action(obs_flat, act_flat).cpu().numpy()  # [N*A, D]
        phi_sa_base = phi_sa_base.reshape(N, A, q_network.rep_dim)     # [N, A, D]

    N_base, A_base, D_base = phi_sa_base.shape
    phi_sa_base_flat = phi_sa_base.reshape(N_base * A_base, D_base)

    print("states_base:", states_base.shape)
    print("coords_base:", coords_base.shape)
    print("phi(s,a) base:", phi_sa_base.shape)
    print("psi(z) base:", psi_z_base.shape)

    # Convert to torch
    phisa_t = torch.tensor(phi_sa_base_flat, dtype=torch.float32)
    psi_t   = torch.tensor(psi_z_base.squeeze(0), dtype=torch.float32)  # [D]

    # 1) Norm statistics
    norms = phisa_t.norm(dim=-1)   # [N*A]
    print(f"phi(s,a) norms: mean={norms.mean().item():.4f}, std={norms.std().item():.4f}")

    # 2) Effective rank (via singular values)
    phisa_centered = phisa_t - phisa_t.mean(dim=0, keepdim=True)
    U, S, Vh = torch.linalg.svd(phisa_centered, full_matrices=False)
    S_np = S.cpu().numpy()
    explained = (S_np ** 2) / (S_np ** 2).sum()
    cumulative = explained.cumsum()

    print("top 5 singular values:", S_np[:5])
    print("cumulative variance (first 5 dims):", cumulative[:5])

    psi_unit = psi_t / (psi_t.norm() + 1e-8)
    phisa_unit = phisa_t / (phisa_t.norm(dim=-1, keepdim=True) + 1e-8)

    # 2) Compute cosine similarities
    cos = phisa_unit @ psi_unit  # [N*A]
    cos_np = cos.cpu().numpy()

    print(f"cos(phi(s,a), psi): mean={cos_np.mean():.4f}, std={cos_np.std():.4f}")

    # 3) Create side-by-side plots
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Left: Variance spectrum
    axes[0].plot(cumulative, marker="o")
    axes[0].set_xlabel("Number of components")
    axes[0].set_ylabel("Cumulative variance explained")
    axes[0].set_title(f"Variance spectrum of phi(s,a) [{goal}]")
    axes[0].grid(alpha=0.3)

    # Right: Cosine similarity histogram
    axes[1].hist(cos_np, bins=40, alpha=0.7)
    axes[1].set_xlabel("cos(phi(s,a), psi(z))")
    axes[1].set_ylabel("count")
    axes[1].set_title(f"Cosine histo phi(s,a) vs psi(z) [{goal}]")
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


    eval_env_first.close()


## Training Loop across goals and seeds

In [ ]:
SEEDS = [42]
GOALS = [(7, 7), (17,13)]

overall_results = {
    goal: {
        "eval_returns": [],
        "eval_returns_time": [],
        "min_steps": [],
        "min_time": [],
        "task_embeddings": [],
        "sa_embeddings": [],
        "sa_fixed_probe_embeddings": [],
        "sa_batches_final": [],
    }
    for goal in GOALS
}

for seed in SEEDS:
    print(f"\n================ SEED {seed} ================\n")
    set_seed(seed)  # your helper

    # Create env to get obs_dim, num_actions once
    env_tmp = make_env(goal=GOALS[0])
    obs_dim = env_tmp.observation_space.shape[0]
    num_actions = env_tmp.action_space.n
    env_tmp.close()

    # One network per seed, reused across goals
    q_net = FactorisedDQN_QNetwork(
        obs_dim=obs_dim,
        num_actions=num_actions,
        goal_dim=2,
        hidden_dim=128,
        rep_dim=64,
    ).to(DEVICE)

    q_target = FactorisedDQN_QNetwork(
        obs_dim=obs_dim,
        num_actions=num_actions,
        goal_dim=2,
        hidden_dim=128,
        rep_dim=64,
    ).to(DEVICE)
    q_target.load_state_dict(q_net.state_dict())
    for p in q_target.parameters():
        p.requires_grad_(False)

    base_goal_encoder_state = deepcopy(q_net.goal_encoder.state_dict())
    # base_obs_last_layer_state = deepcopy(q_net.obs_encoder[-1].state_dict())

    # Per-seed task embedding memory (for repulsion)
    seed_task_embedding_memory = []
    seen_goal_labels = []
    prev_reference_params = None
    prev_fisher_diag = None
    prev_goal = None
    prev_buffer = None


    for goal_idx, goal in enumerate(GOALS):
        print(f"\n----- seed={seed}, goal={goal} -----\n")

        # Resync target to current q_net before training on this goal
        q_target.load_state_dict(q_net.state_dict())
        for p in q_target.parameters():
            p.requires_grad_(False)

        # q_net.goal_encoder.load_state_dict(base_goal_encoder_state)
        # q_target.goal_encoder.load_state_dict(base_goal_encoder_state)

        # q_net.obs_encoder[-1].load_state_dict(base_obs_last_layer_state)
        # q_target.obs_encoder[-1].load_state_dict(base_obs_last_layer_state)

        # Decide which params to train (transfer vs. full)
        if goal_idx == 0:
            # First goal: train full network
            for p in q_net.parameters():
                p.requires_grad_(True)
            trainable_params = None  # let dqn_train create its usual optimizer
            regulariser = None
            reference_params = None
            fisher_diag = None
        else:

            for p in q_net.parameters():
                p.requires_grad_(True)
            # Subsequent goals: reuse phi, train only psi (and optionally last phi layer)
            # for p in q_net.sa_encoder.parameters():
            #     p.requires_grad_(False)
            # for p in q_net.goal_encoder.parameters():
            #     p.requires_grad_(True)

            # # If you want psi + last phi layer instead of only psi:
            # last_layer = list(q_net.sa_encoder.children())[-1]
            # for p in last_layer.parameters():
            #     p.requires_grad_(True)

            trainable_params = [p for p in q_net.parameters() if p.requires_grad]
            regulariser = "repulsion"  # or None if you don’t want repulsion here

        # Train on this goal, reusing q_net weights from previous goals
        (
            q_network,
            q_target_network,
            eval_returns,
            eval_returns_time,
            min_steps,
            min_time,
            task_embedding,
            sa_embedding_mean,
            sa_embedding_fixed,
            sa_batch_final,
            buffer,
        )= dqn_train(
            q_network=q_net,
            q_target_network=q_target,
            env=make_env(goal=goal),
            goal=goal,
            device=DEVICE,
            embedding_memory=seed_task_embedding_memory,  # previous task embeddings in this seed
            regulariser=regulariser,
            reg_alpha=1,          # your choice 
            params=trainable_params, # None for full training, list for transfer mode
            reference_params=prev_reference_params,
            fisher_diag=prev_fisher_diag,
            sa_reg_prefix_filter="sa_encoder",
        )
        visualise_q_table(goal, q_net, eval_returns=eval_returns)
        visualise_embeddings(goal, q_net)   
        seed_task_embedding_memory.append(task_embedding)
        seen_goal_labels.append(str(goal))
        print_goal_embedding_similarity(seed_task_embedding_memory, goal_labels=seen_goal_labels)

        prev_reference_params = snapshot_named_parameters(q_net, prefix_filter="sa_encoder")
        prev_fisher_diag = estimate_fisher_diag(
                model=q_net,
                target_model=q_target,
                replay_buffer=buffer,
                goal=goal,
                num_actions=num_actions,
                device=DEVICE,
                gamma=0.99,
                batch_size=256,
                n_batches=64,
                prefix_filter="sa_encoder"
        )

        ewc_param_count = sum(
            p.numel() for name, p in q_net.named_parameters()
            if name in prev_reference_params
        )
        total_param_count = sum(p.numel() for p in q_net.parameters())

        print("EWC params:", ewc_param_count, "Total params:", total_param_count)

        print("ref len:", len(prev_reference_params))
        if prev_fisher_diag is not None:
            print("fisher len:", len(prev_fisher_diag))
            some_key = next(iter(prev_fisher_diag.keys()))
            print("sample F value:", prev_fisher_diag[some_key].abs().mean().item())

        prev_goal = goal
        prev_buffer = buffer

        overall_results[goal]["eval_returns"].append(eval_returns)
        overall_results[goal]["eval_returns_time"].append(eval_returns_time)
        overall_results[goal]["min_steps"].append(min_steps)
        overall_results[goal]["min_time"].append(min_time)
        overall_results[goal]["task_embeddings"].append(task_embedding)
        overall_results[goal]["sa_embeddings"].append(sa_embedding_mean)
        overall_results[goal]["sa_fixed_probe_embeddings"].append(sa_embedding_fixed)
        overall_results[goal]["sa_batches_final"].append(sa_batch_final)

## Backward Compatibility test (or new goal test)

Do not retrain anything, just give the task embedding, and check whether the sa encoder can give good Q functions still. The task embedding must be previously learned.

In [ ]:
GOALS = [(9, 9)]

def compute_embedding_drift(overall_results, q_net, device):
    drift = {}
    for goal, data in overall_results.items():
        if len(data["task_embeddings"]) == 0:
            continue

        # e.g. embedding right after this goal was learned
        emb_old = np.asarray(data["task_embeddings"][0], dtype=np.float32)

        # embedding now, with final goal encoder
        goal_arr = np.array(goal, dtype=np.float32)
        goal_t = torch.tensor(goal_arr, dtype=torch.float32, device=device).unsqueeze(0)
        with torch.no_grad():
            emb_new = q_net.encode_goal(goal_t).squeeze(0).cpu().numpy()

        # cosine similarity
        def _norm(x):
            return x / (np.linalg.norm(x) + 1e-8)

        cos = float(np.dot(_norm(emb_old), _norm(emb_new)))
        drift[goal] = cos
    return drift

drift = compute_embedding_drift(overall_results, q_net, DEVICE)
for goal, cos in drift.items():
    print(goal, "cos(old, final) =", cos)

for goal in GOALS:
    task_embedding = overall_results[goal]["task_embeddings"]
    visualise_q_table(goal, q_net, eval_returns=None, task_embedding=task_embedding)
    visualise_embeddings(goal, q_net)

## HTML visualisation

In [ ]:
from visualisations import plot_full_embedding_dashboard_html

for goal, data in overall_results.items():
    print(goal)
    print("task_embeddings:", len(data.get("task_embeddings", [])))
    print("sa_fixed_probe_embeddings:", len(data.get("sa_fixed_probe_embeddings", [])))
    print("sa_batches_final:", len(data.get("sa_batches_final", [])))

dashboard = plot_full_embedding_dashboard_html(
    overall_results=overall_results,
    qnet=q_net,
    mode="all",
    save_html="plots/full_embedding_dashboard_maze_large.html",
)

print(dashboard["save_html"])
print(dashboard["isotropy_metrics"])